# WasteWise AI — reproducible Colab training

This notebook trains a real six-class model only from a dataset you have inspected and have permission to use. It does not contain, download, or invent a dataset or model. In Colab, open **File → Upload notebook** and select this `.ipynb`, then choose **Runtime → Change runtime type → T4 GPU** (or another available accelerator). The workflow automatically falls back to CPU.

**Do not train until the dataset has independently verified labels and license for all six classes:** `plastic`, `paper`, `metal`, `glass`, `organic`, `other`. A class named `trash` is not a justified substitute for `other`; do not remap it. Any missing or ambiguous class stops the notebook.

In [ ]:
import os, platform, sys
print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch, torchvision
    print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
    else:
        print('Using CPU; training will be slower.')
except ImportError as exc:
    raise RuntimeError('Select a standard Google Colab runtime with PyTorch/torchvision installed.') from exc

## Install only export/evaluation dependencies

Colab supplies a PyTorch/torchvision build matched to its CUDA runtime. Keep that build rather than replacing it with a generic pip wheel. Restart the runtime if Colab requests it, then rerun the cells from the top.

In [ ]:
%pip install -q "Pillow>=11,<13" "numpy>=1.26,<3" "scikit-learn>=1.5,<2" "onnx>=1.17,<2" "onnxruntime>=1.20,<2"

## Dataset source and license gate

## Recommended candidate: UCI RealWaste

The [UCI RealWaste record](https://archive.ics.uci.edu/dataset/908/realwaste) documents an authentic landfill image-classification dataset, created in 2023 by Sam Single, Saeid Iranmanesh, and Raad Raad, with 4,752 images and CC BY 4.0 terms. UCI's [official metadata](https://archive.ics.uci.edu/api/dataset?id=908) reports these original labels/counts: Cardboard 461, Food Organics 411, Glass 420, Metal 790, Miscellaneous Trash 495, Paper 500, Plastic 921, Textile Trash 318, Vegetation 436. Its official download is [realwaste.zip](https://archive.ics.uci.edu/static/public/908/realwaste.zip). This project has not downloaded or inspected the archive; verify current UCI terms, contents, class names and image rights before using it, and cite the dataset and its authors.

The nine UCI labels are **not** the six target labels. A candidate grouping to inspect and explicitly approve is `Cardboard` + `Paper` → `paper`; `Food Organics` + `Vegetation` → `organic`; `Miscellaneous Trash` + `Textile Trash` → `other`; and the remaining three map directly. These are human relabeling choices, not source-provided equivalences. In particular, assess whether vegetation belongs in the local organic category and whether textiles belong in `other`; if the application category definitions do not justify them, collect/label suitable images instead. Do not use this mapping without manual inspection and approval. The notebook records original-to-target mappings in model metadata and rejects missing target categories.

The public [TrashNet authors' repository](https://github.com/garythung/trashnet) describes 2,527 controlled-background photos with `glass`, `paper`, `cardboard`, `plastic`, `metal`, and `trash`; it has no organic class, and the meaning/coverage of `trash` is not a safe substitute for `other`. Its dataset-image licensing should be confirmed separately from its software license before reuse. TrashNet alone is not a six-class solution.

UCI metadata does not provide object/scene group IDs or recommended train/validation/test splits. The ZIP must therefore be manually curated before training: retain the original source class names, put related views/captures of the same object or scene in one group directory, and never create a separate group per image just to satisfy the folder layout. The notebook stops if a source class is flat rather than grouped.

```text
waste-six-class/
  plastic/<unique-group-name>/*.(jpg|jpeg|png|webp)
  paper/<unique-group-name>/*.(jpg|jpeg|png|webp)
  metal/<unique-group-name>/*.(jpg|jpeg|png|webp)
  glass/<unique-group-name>/*.(jpg|jpeg|png|webp)
  organic/<unique-group-name>/*.(jpg|jpeg|png|webp)
  other/<unique-group-name>/*.(jpg|jpeg|png|webp)
```

There must be at least seven non-empty independent groups in each source class so train/validation/test splits can be made without splitting a group. Prepare a copy of the source data in this layout after an explicitly reviewed mapping; do not overwrite or discard source labels. The inspection step reports imbalance and will stop on missing labels, corrupt files, duplicates across groups, suspicious near duplicates, or too few groups.

In [ ]:
# Choose 'upload' for a ZIP from your computer or 'drive' for a file in Google Drive.
SOURCE_MODE = 'upload'
DRIVE_ZIP_PATH = '/content/drive/MyDrive/WasteWiseAI/waste-six-class.zip'
WORK_DIR = '/content/wastewise-training'

from pathlib import Path
import shutil, zipfile

work_dir = Path(WORK_DIR).resolve()
work_dir.mkdir(parents=True, exist_ok=True)
if SOURCE_MODE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1:
        raise ValueError('Upload exactly one dataset ZIP file.')
    zip_path = Path('/content') / zip_names[0]
elif SOURCE_MODE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    zip_path = Path(DRIVE_ZIP_PATH)
    if not zip_path.is_file():
        raise FileNotFoundError(f'Dataset ZIP not found: {zip_path}')
else:
    raise ValueError("SOURCE_MODE must be 'upload' or 'drive'.")

extract_dir = work_dir / 'extracted'
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir()
with zipfile.ZipFile(zip_path) as archive:
    root = extract_dir.resolve()
    for member in archive.infolist():
        destination = (extract_dir / member.filename).resolve()
        if destination != root and root not in destination.parents:
            raise ValueError('The ZIP contains a path outside its extraction directory.')
    archive.extractall(extract_dir)

CATEGORIES = ('plastic', 'paper', 'metal', 'glass', 'organic', 'other')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}
SOURCE_LABEL_MAPPING = {
    'Cardboard': 'paper', 'Paper': 'paper',
    'Food Organics': 'organic', 'Vegetation': 'organic',
    'Glass': 'glass', 'Metal': 'metal', 'Plastic': 'plastic',
    'Miscellaneous Trash': 'other', 'Textile Trash': 'other',
}
MAPPING_REVIEWED_AND_APPROVED = False
DATASET_NAME = 'UCI RealWaste (verify archive and current terms before use)'
DATASET_LICENSE = 'CC BY 4.0 as stated by UCI; verify current license and image rights'
DATASET_PROVENANCE = 'https://archive.ics.uci.edu/dataset/908/realwaste; https://archive.ics.uci.edu/api/dataset?id=908'
def normalize_label(value):
    return ''.join(character.lower() for character in value if character.isalnum())
mapping_by_normalized_label = {normalize_label(source): target for source, target in SOURCE_LABEL_MAPPING.items()}
roots = []
for candidate in [extract_dir, *[p for p in extract_dir.rglob('*') if p.is_dir()]]:
    child_dirs = [path for path in candidate.iterdir() if path.is_dir()]
    if {normalize_label(path.name) for path in child_dirs} == set(mapping_by_normalized_label):
        roots.append((candidate, child_dirs))
if len(roots) != 1:
    candidates = []
    for candidate in [extract_dir, *[p for p in extract_dir.rglob('*') if p.is_dir()]]:
        labels = {normalize_label(path.name) for path in candidate.iterdir() if path.is_dir()}
        if len(labels & set(mapping_by_normalized_label)) >= 3:
            missing_source = sorted(set(mapping_by_normalized_label) - labels)
            extra_source = sorted(labels - set(mapping_by_normalized_label))
            candidates.append((str(candidate), missing_source, extra_source))
    raise ValueError(f'Expected one original-label root. Candidate folders and missing/extra source labels: {candidates[:5]}')
RAW_DATASET_DIR, source_class_dirs = roots[0]
print('Original dataset root:', RAW_DATASET_DIR)
print('Original labels and image counts:')
for source_dir in sorted(source_class_dirs):
    count = sum(1 for path in source_dir.rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
    group_folders = [path for path in source_dir.iterdir() if path.is_dir()]
    print(f'{source_dir.name}: {count} images, {len(group_folders)} candidate group folders')

# Visually inspect representative images under their original source labels before approving any mapping.
from IPython.display import display
from PIL import Image
for source_dir in sorted(source_class_dirs):
    samples = [path for path in source_dir.rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS][:3]
    print('Original class:', source_dir.name)
    for sample in samples:
        with Image.open(sample) as image:
            print(sample.relative_to(RAW_DATASET_DIR))
            display(image.convert('RGB').resize((224, 224)))
# Preserve object/scene groups. Flat class folders are rejected; create group subfolders manually first.
import csv, hashlib
DATASET_DIR = work_dir / 'prepared-six-class'
if DATASET_DIR.exists(): shutil.rmtree(DATASET_DIR)
for category in CATEGORIES: (DATASET_DIR / category).mkdir(parents=True, exist_ok=True)
used_group_names = set()
with (DATASET_DIR / 'source-labels.csv').open('w', newline='', encoding='utf-8') as audit_file:
    writer = csv.writer(audit_file)
    writer.writerow(['prepared_relative_path', 'original_label', 'target_label', 'original_relative_path', 'group_id'])
    for source_dir in sorted(source_class_dirs):
        original_label = source_dir.name
        target_label = SOURCE_LABEL_MAPPING[next(name for name in SOURCE_LABEL_MAPPING if normalize_label(name) == normalize_label(original_label))]
        group_dirs = [path for path in source_dir.iterdir() if path.is_dir()]
        if not group_dirs or any(path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS for path in source_dir.iterdir()):
            raise ValueError(f'{source_dir.name} must contain only reviewed object/scene group subfolders, not flat images.')
        for group_dir in sorted(group_dirs):
            group_id = group_dir.name
            if group_id in used_group_names:
                raise ValueError(f'Group ID {group_id!r} appears in multiple source classes; consolidate related items to prevent split leakage.')
            used_group_names.add(group_id)
            group_images = [path for path in group_dir.rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS]
            target_group = DATASET_DIR / target_label / group_id
            target_group.mkdir(parents=True, exist_ok=True)
            for source_path in sorted(group_images):
                relative = source_path.relative_to(RAW_DATASET_DIR)
                destination = target_group / source_path.name
                if destination.exists():
                    destination = target_group / (hashlib.sha256(str(relative).encode()).hexdigest()[:12] + source_path.suffix.lower())
                shutil.copy2(source_path, destination)
                writer.writerow([str(destination.relative_to(DATASET_DIR)), original_label, target_label, str(relative), group_id])
print('Prepared target dataset:', DATASET_DIR)
print('Original labels and groups retained in source-labels.csv')

In [ ]:
# Inspect labels, group counts, image readability, exact duplicates, and near duplicates.
from collections import Counter, defaultdict
from PIL import Image, ImageFile
import hashlib
import numpy as np

ImageFile.LOAD_TRUNCATED_IMAGES = False
class_dirs = {p.name for p in DATASET_DIR.iterdir() if p.is_dir()}
missing = sorted(set(CATEGORIES) - class_dirs)
unsupported = sorted(class_dirs - set(CATEGORIES))
if missing or unsupported:
    raise ValueError(f'Class folders must match exactly. Missing: {missing}; unsupported: {unsupported}. Do not map labels silently.')

records, corrupt = [], []
group_names = set()
for label in CATEGORIES:
    category_dir = DATASET_DIR / label
    for group_dir in sorted(category_dir.iterdir()):
        if not group_dir.is_dir():
            raise ValueError(f'Put images in group subfolders; unexpected file/folder: {group_dir}')
        if group_dir.name in group_names:
            raise ValueError(f'Group names must be globally unique: {group_dir.name}')
        group_names.add(group_dir.name)
        for path in sorted(group_dir.rglob('*')):
            if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            try:
                with Image.open(path) as image:
                    image.verify()
                with Image.open(path) as image:
                    image.load()
                    if image.width * image.height > 40_000_000:
                        raise ValueError('image has more than 40 million pixels')
                raw = path.read_bytes()
                with Image.open(path) as image:
                    pixels = np.asarray(image.convert('L').resize((9, 8), Image.Resampling.BILINEAR))
                bits = (pixels[:, 1:] > pixels[:, :-1]).reshape(-1)
                dhash = sum(int(bit) << index for index, bit in enumerate(bits))
                records.append({'path': path, 'label': label, 'group': group_dir.name,
                                'sha256': hashlib.sha256(raw).hexdigest(), 'dhash': dhash})
            except Exception as exc:
                corrupt.append((str(path), str(exc)))

counts = Counter(item['label'] for item in records)
group_counts = {label: len({item['group'] for item in records if item['label'] == label}) for label in CATEGORIES}
print('Readable image counts:', dict(counts))
print('Independent group counts:', group_counts)
print('Unreadable/oversized files:', len(corrupt))
if corrupt:
    print('Examples:', corrupt[:10])
    raise ValueError('Remove or replace corrupt/oversized files, then rerun the inspection.')
if any(counts[label] < 10 or group_counts[label] < 7 for label in CATEGORIES):
    raise ValueError('Every target class needs at least 10 readable images across at least seven independent groups.')
print('Largest/smallest class image-count ratio:', max(counts.values()) / min(counts.values()))

exact = defaultdict(list)
for item in records:
    exact[item['sha256']].append(item)
exact_conflicts = [items for items in exact.values() if len({(x['label'], x['group']) for x in items}) > 1]
if exact_conflicts:
    print('Exact duplicate conflicts:', [[str(x['path']) for x in group] for group in exact_conflicts[:10]])
    raise ValueError('Exact duplicates cross groups/classes; deduplicate or group the same source object together.')

# Four 16-bit bands cheaply find candidate 64-bit dHash pairs; distance <= 3 is flagged.
buckets = defaultdict(list)
near_conflicts = []
for index, item in enumerate(records):
    value = item['dhash']
    candidates = set()
    for band in range(4):
        bucket_key = (band, (value >> (band * 16)) & 0xffff)
        candidates.update(buckets[bucket_key])
    for other_index in candidates:
        other = records[other_index]
        distance = (value ^ other['dhash']).bit_count()
        if distance <= 3 and (item['label'], item['group']) != (other['label'], other['group']):
            near_conflicts.append((str(other['path']), str(item['path']), distance,
                                   other['label'], item['label']))
    for band in range(4):
        buckets[(band, (value >> (band * 16)) & 0xffff)].append(index)
print('Near-duplicate cross-group pairs (dHash distance <= 3):', len(near_conflicts))
if near_conflicts:
    print('Examples:', near_conflicts[:20])
    raise ValueError('Review near duplicates and put images of the same item/scene in one group or remove duplicates.')
print('Dataset inspection passed. Near-duplicate hashing is a practical screen, not proof of no leakage.')

## Reproducible group split and transfer learning

The training script splits each category's independent group directories using seed `42` into approximately 70% train, 15% validation, and 15% held-out test. A group remains wholly in one split; the test set is used only after validation-based early stopping. It fine-tunes torchvision ResNet18 from explicit `ResNet18_Weights.IMAGENET1K_V1` weights, balances the training loss by class frequency, and uses CUDA/AMP when available. The model graph embeds ImageNet normalization so inference still accepts the existing raw RGB `[0,1]` ONNX input contract. The validation/test metrics below will be computed by the cells; no results are prefilled.

For recovery, mount Drive and save the checkpoint there. The script atomically replaces its checkpoint after each completed epoch. If the runtime disconnects, reconnect, repeat setup/data cells, set `RESUME_FROM` to the checkpoint path, and rerun training. A partial epoch may be repeated.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import json, os, subprocess

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
if not MAPPING_REVIEWED_AND_APPROVED:
    raise ValueError('Review source labels, class definitions, and license; set MAPPING_REVIEWED_AND_APPROVED=True before training.')
PROJECT_DIR = Path('/content/wastewise-project')
training_script = PROJECT_DIR / 'training' / 'train_colab_model.py'
if not training_script.is_file():
    print('Upload ai-service/training/train_colab_model.py from the project.')
    uploaded_script = files.upload()
    script_names = [name for name in uploaded_script if name.endswith('train_colab_model.py')]
    if len(script_names) != 1:
        raise ValueError('Upload exactly the project train_colab_model.py script.')
    training_script.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(Path('/content') / script_names[0], training_script)
OUTPUT_DIR = Path('/content/drive/MyDrive/WasteWiseAI/model-output')
CHECKPOINT = Path('/content/drive/MyDrive/WasteWiseAI/checkpoints/last-checkpoint.pt')
RESUME_FROM = None  # Set to str(CHECKPOINT) after a disconnected run.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

command = [
    sys.executable, str(training_script),
    '--dataset-dir', str(DATASET_DIR),
    '--dataset-name', DATASET_NAME,
    '--dataset-license', DATASET_LICENSE,
    '--dataset-provenance', DATASET_PROVENANCE,
    '--source-label-mapping-json', json.dumps(SOURCE_LABEL_MAPPING),
    '--output-dir', str(OUTPUT_DIR),
    '--checkpoint', str(CHECKPOINT),
    '--epochs', '12', '--batch-size', '32', '--patience', '4',
]
if RESUME_FROM:
    command.extend(['--resume-from', RESUME_FROM])
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
# Verify the exported file using the serving runtime and inspect the measured held-out report.
import json, onnx, onnxruntime as ort
import numpy as np
from PIL import Image
from torchvision.transforms import InterpolationMode, Resize, ToTensor

model_path = OUTPUT_DIR / 'waste-classifier.onnx'
manifest_path = OUTPUT_DIR / 'model-manifest.json'
evaluation_path = OUTPUT_DIR / 'evaluation.json'
manifest = json.loads(manifest_path.read_text())
evaluation = json.loads(evaluation_path.read_text())
assert manifest['class_labels'] == list(CATEGORIES)
assert manifest['input_size'] == 224 and manifest['input_layout'] == 'NCHW'
assert manifest['input_range'] == '0_1' and manifest['output_type'] == 'logits'
assert manifest['preprocessing']['normalization'] == 'imagenet_embedded'
onnx.checker.check_model(onnx.load(str(model_path)))
session = ort.InferenceSession(str(model_path), providers=['CPUExecutionProvider'])
assert len(session.get_inputs()) == 1 and len(session.get_outputs()) == 1
assert session.get_inputs()[0].shape[1:] == [3, 224, 224]
assert session.get_outputs()[0].shape[1] == 6
smoke_path = next(item['path'] for item in records if item['label'] == manifest['class_labels'][0])
with Image.open(smoke_path) as image:
    tensor = ToTensor()(Resize((224, 224), interpolation=InterpolationMode.BILINEAR)(image.convert('RGB')))
logits = session.run(None, {'images': tensor.unsqueeze(0).numpy()})[0]
assert logits.shape == (1, 6) and np.isfinite(logits).all()
scores = np.exp(logits[0] - logits[0].max())
scores = scores / scores.sum()
print('ONNX inference smoke test passed; actual model output (not a claimed accuracy):')
print({'predicted_category': CATEGORIES[int(scores.argmax())], 'confidence': float(scores.max())})
print('Held-out test evaluation generated by the training run:')
print(json.dumps(evaluation, indent=2))

In [ ]:
# Bundle deployment artifacts and download them; split manifests can contain source filenames, so review before sharing.
import shutil
from google.colab import files

required_artifacts = [
    OUTPUT_DIR / 'waste-classifier.onnx',
    OUTPUT_DIR / 'model-manifest.json',
    OUTPUT_DIR / 'training-config.json',
    OUTPUT_DIR / 'evaluation.json',
    OUTPUT_DIR / 'split-manifest.json',
]
missing = [str(path) for path in required_artifacts if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Missing model artifacts: {missing}')
shutil.copy2(DATASET_DIR / 'source-labels.csv', OUTPUT_DIR / 'source-labels.csv')
archive_base = '/content/WasteWiseAI-model-artifacts'
archive_path = shutil.make_archive(archive_base, 'zip', OUTPUT_DIR)
print('Download this archive and inspect its provenance/evaluation files before deployment.')
files.download(archive_path)

## Install the model locally

Extract the archive and copy `waste-classifier.onnx` and `model-manifest.json` to `ai-service/models/` in the project. Keep the evaluation/training/split files with your experiment records. The manifest label order and embedded preprocessing are checked by the service. Restart FastAPI, then check `http://127.0.0.1:8000/health`; require `model.ready: true` before trying the citizen page. Run the AI service tests and an authenticated classification request. Do not publish the model or dataset unless their licenses permit it. This notebook contains no measured training results until its training/evaluation cells have run.